In [1]:
# Imports
import sys
import logging
from datetime import datetime

sys.path.insert(0, '../../../LOGOS')
from src import Pert, plot_gantt_chart, plot_resource_utilization, plot_location_utilization, plot_equipment_utilization
# Configure logging in the runner (avoid setting basicConfig inside the module)
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
import json

In [ ]:
json_path = "example_10.json"
sgs = "max_use_res_ranked"
gantt_file = "gantt.html"
csv_file = "schedule.csv"
schema_file = "outage_schema.json"

resource_type = "MECHANIC"
resource_plot_file = "mechanics.html"
location_id = "LOC_REACTOR_CAVITY"
location_plot_file = "cavity.html"


# 1) Load data & build schedule graph
pert = Pert.from_json_file(json_path, schema_path=schema_file)

# 1.1) debug situations with schedule
pert.debug_connectivity_and_es()
pert.debug_candidates_and_capacity(hours_ahead=48)

# 1.2) Perform clasical CPM calculation
pert.generateInfo()
logging.info(f"CPM Duration: {pert.getProjectDuration():.1f} hours")

# 2) Run RCPSP with capacity-aware selection
logging.info(f"Scheduling strategy: {sgs}")
results = pert.calculateScheduleWithResources(sgs=sgs, max_time_hours=24*7)
pert.print_chain_sets_summary()
pert.explain_idle_on_chain()

print("================")
pert.explain_idle_on_chain_detailed()

logging.info(f"RCPSP Completed: {results['n_completed']}/{results['n_activities']}")
logging.info(f"Actual Duration: {results['scheduled_duration']:.1f} hours")
logging.info(f"Total Delay: {results['delay_hours']:.1f} hours")

# 3) Print a summary (do this FIRST so you see output even if export/plots fail)
try:
    pert.print_schedule_summary()
except Exception as e:
    logging.error(f"Error while printing schedule summary: {e}")

# 4) Export CSV (guarded)
try:
    pert.export_schedule_to_csv(csv_file)
    logging.info(f"CSV exported: {csv_file}")
except Exception as e:
    logging.error(f"CSV export failed: {e}")

# 5) Generate plots (guarded)
try:
    plot_gantt_chart(pert, filename=gantt_file, show_delays=True)
    logging.info(f"Gantt chart: {gantt_file}")
except Exception as e:
    logging.error(f"Gantt chart failed: {e}")

for resource_type in pert.crew_pool.get_all_skills():
    resource_plot_file = str(resource_type) + ".html"
    try:
        plot_resource_utilization(pert, resource_type, filename=resource_plot_file)
        logging.info(f"Resource utilization ({resource_type}): {resource_plot_file}")
    except Exception as e:
        logging.error(f"Resource utilization plot failed: {e}")

for location_id in pert.location_pool.get_all_location_ids():
    location_plot_file = str(location_id) + ".html"
    try:
        plot_location_utilization(pert, location_id, filename=location_plot_file)
        logging.info(f"Location utilization ({location_id}): {location_plot_file}")
    except Exception as e:
        logging.error(f"Location utilization plot failed: {e}")

for equipment_id in pert.equipment_pool.get_all_equipment_ids():
    equipment_plot_file = str(equipment_id) + ".html"
    try:
        plot_equipment_utilization(pert, equipment_id, filename=equipment_plot_file)
        logging.info(f"Equipment utilization ({equipment_id}): {equipment_plot_file}")
    except Exception as e:
        logging.error(f"Equipment utilization plot failed: {e}")

# 6) Extra diagnostics: show a few steps from the scheduling log
if hasattr(pert, "schedule_log") and pert.schedule_log:
    head = pert.schedule_log[:5]
    logging.info("First scheduling steps (diagnostic):")
    for step in head:
        logging.info(
            "t=%s candidates=%s selected=%s ongoing=%s n_completed=%s",
            step['time'].strftime('%Y-%m-%d %H:%M'),
            step['candidates'],
            step['selected'],
            step['ongoing'],
            step['n_completed']
        )
else:
    logging.info("No schedule_log available or empty.")

In [3]:
# DAG visualization
pert.plot_activity_dag(
    filename="dag_augmented_plotly.html",
    library="plotly",
    highlight="constrained",     # colour the resource-constrained chain
    layer_by="topo",             # stable topological ranks
    include_augmented_edges=True,
    show_unscheduled=False,
    show_edge_arrows=True,
)

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'hoverinfo': 'none',
              'line': {'color': '#95a5a6', 'width': 1},
              'mode': 'lines',
              'type': 'scatter',
              'x': [-0.6, -0.6, None, -0.6, -0.6, None, -0.6, -0.6, None, -0.6,
                    -0.6, None, -0.6, -1.2, None, -0.6, 0.0, None, -1.2, -2.4,
                    None, -2.4, -1.2, None, 0.0, -1.2, None, 0.0, 0.0, None, 0.0,
                    1.2, None, -1.2, 0.0, None, 0.0, 0.0, None, 1.2, 0.0, None,
                    -1.2, -0.6, None, 0.0, -0.6, None, -0.6, -0.6, None],
              'y': [-0.0, -1.0, None, -1.0, -2.0, None, -2.0, -3.0, None, -3.0,
                    -4.0, None, -4.0, -5.0, None, -4.0, -5.0, None, -5.0, -6.0,
                    None, -6.0, -7.0, None, -5.0, -6.0, None, -5.0, -6.0, None,
                    -5.0, -6.0, None, -6.0, -7.0, None, -6.0, -7.0, None, -6.0,
                    -7.0, None, -7.0, -8.0, None, -7.0, -8.0, None, -8.0, -9.0,
                    None]},
             {'hoverinfo': 'none',
              'line': {'color': '#c0392b', 'dash': 'dash', 'width': 2},
              'mode': 'lines',
              'name': 'Augmented (→)',
              'type': 'scatter',
              'x': [-2.4, 1.2, None],
              'y': [-6.0, -6.0, None]},
             {'hoverinfo': 'text',
              'hovertext': [<b>START</b><br><b>Descr:</b> Outage start
                            (anchor)<br><b>Duration:</b> 1.0 h<br><b>Delay:</b> 0.0
                            h<br><b>CPM:</b> ES=0.0, EF=1.0, LS=0.0,
                            LF=1.0<br><b>Slack:</b> 0.0 h<br><b>Actual TF:</b> 0.00
                            h<br><b>Start:</b> 2025-09-01 00:00<br><b>End:</b>
                            2025-09-01 01:00<br><b>Location:</b> —<br><b>Chain
                            flags:</b> CPM, Constrained, Zero TF
                            actual<br><b>Resources:</b> None<br><b>Equipment:</b>
                            None<br>, <b>P001</b><br><b>Descr:</b> Outage
                            mobilization — crew check-in, safety briefings, ALARA
                            training, and radiation work permit
                            preparation<br><b>Duration:</b> 4.0 h<br><b>Delay:</b>
                            0.0 h<br><b>CPM:</b> ES=1.0, EF=5.0, LS=1.0,
                            LF=5.0<br><b>Slack:</b> 0.0 h<br><b>Actual TF:</b> 0.00
                            h<br><b>Start:</b> 2025-09-01 01:00<br><b>End:</b>
                            2025-09-01 05:00<br><b>Location:</b> —<br><b>Chain
                            flags:</b> CPM, Constrained, Zero TF
                            actual<br><b>Resources:</b> None<br><b>Equipment:</b>
                            None<br>, <b>P002</b><br><b>Descr:</b> Equipment,
                            rigging, and scaffolding material staging — complete
                            tool kit assembly for all containment work
                            packages<br><b>Duration:</b> 20.0 h<br><b>Delay:</b>
                            0.0 h<br><b>CPM:</b> ES=5.0, EF=25.0, LS=5.0,
                            LF=25.0<br><b>Slack:</b> 0.0 h<br><b>Actual TF:</b>
                            0.00 h<br><b>Start:</b> 2025-09-01 05:00<br><b>End:</b>
                            2025-09-02 01:00<br><b>Location:</b> —<br><b>Chain
                            flags:</b> CPM, Constrained, Zero TF
                            actual<br><b>Resources:</b> MECHANIC:
                            2<br><b>Equipment:</b> None<br>,
                            <b>P003</b><br><b>Descr:</b> Polar crane pre-
                            operational check (exterior) — wire rope inspection,
                            limit switch test, lubrication, and pre-lift rigging
                            hardware setup<br><b>Duration:</b> 7.0
                            h<br><b>Delay:</b> 0.0 h<br><b>CPM:</b> ES=25.0,
                            EF=32.0, LS=25.0, LF=32.0<br><b>Slack:</b> 0.0
                            h<br><b>A

In [ ]:
# Testing serial SGS

# Load
pert = Pert.from_json_file(json_path, schema_path=schema_file)

# Run Serial SGS with a chosen priority rule
results = pert.calculateSerialScheduleWithResources(priority_rule='lf')

# Inspect results
print(results)

# View schedule
df = pert.get_schedule_dataframe()
print(df[['activity_id', 'start_time', 'end_time', 'duration', 'delay']])

# Gantt chart — semicolon suppresses inline display (avoids nbformat render error)
plot_gantt_chart(pert, filename='serial_gantt.html');

In [5]:
for a, b in pert.forwardDict.items():
  print(b)


[Activity('P001': Outage mobilization — crew check-in, safety briefings, ALARA training, and radiation work permit preparation, duration=4.0h)]
[Activity('P002': Equipment, rigging, and scaffolding material staging — complete tool kit assembly for all containment work packages, duration=20.0h)]
[Activity('P003': Polar crane pre-operational check (exterior) — wire rope inspection, limit switch test, lubrication, and pre-lift rigging hardware setup, duration=7.0h)]
[Activity('C101': Containment access prep & initial radiation survey, duration=6.0h)]
[Activity('C102': Polar crane pre-use inspection, duration=8.0h), Activity('C107': Install temporary scaffolding under containment dome, duration=10.0h)]
[Activity('C103': Remove equipment hatch shielding (lift), duration=10.0h)]
[Activity('C109': Restore equipment hatch shielding (lift), duration=10.0h)]
[Activity('C104': Containment liner visual inspection, duration=8.0h), Activity('C105': ECCS injection header inspection (inside containmen